# Simulating the ABQLBM with a `YMonomial` object

In [ ]:
from qiskit_aer import AerSimulator

from qlbm.components import (
    EmptyPrimitive,
)
from qlbm.components.ab.ab import ABQLBM
from qlbm.components.ab.initial import ABDiscreteUniformInitialConditions
from qlbm.components.ab.measurement import ABGridMeasurement
from qlbm.infra import QiskitRunner, SimulationConfig
from qlbm.lattice import ABLattice
from qlbm.tools.utils import create_directory_and_parents

lattice = ABLattice(
    {
        "lattice": {"dim": {"x": 8, "y": 16}, "velocities": "D2Q9"},
        "geometry": [
            {
                "shape": "ymonomial",
                "exponent": 2,
                "comparator": "<",
                "boundary": "bounceback",
            }
        ],
    }
)

output_dir = "qlbm-output/ab-d2q9-64x32-1-obstacle-ymonomial-qiskit"
create_directory_and_parents(output_dir)

cfg = SimulationConfig(
    initial_conditions=ABDiscreteUniformInitialConditions(
        lattice,
        [1, 5, 8],
        ([], list(range(lattice.num_gridpoints[1].bit_length()))),
    ),
    algorithm=ABQLBM(lattice, True),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABGridMeasurement(lattice),
    target_platform="QISKIT",
    compiler_platform="QISKIT",
    optimization_level=0,
    statevector_sampling=True,
    execution_backend=AerSimulator(method="statevector"),
    sampling_backend=AerSimulator(method="statevector"),
)

cfg.prepare_for_simulation()

In [ ]:
# Number of shots to simulate for each timestep when running the circuit
NUM_SHOTS = 2**12


# Number of timesteps to simulate
NUM_STEPS = 20

runner = QiskitRunner(
    cfg,
    lattice,
)


# Simulate the circuits using both snapshots
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)